In [ ]:
import re
import pandas as pd
from typing import Dict


In [ ]:
class TextPreprocessor:

    def clean_text(self, text: str) -> str:
        if pd.isna(text) or not text:
            return ""

        text = str(text).lower()

        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)

        # Remove URLs
        text = re.sub(r'http\S+|www\.\S+', '', text)

        # Remove emails
        text = re.sub(r'\S+@\S+', '', text)

        # Remove unwanted characters
        text = re.sub(r'[^a-zA-Z0-9\s\.,!?+#-]', ' ', text)

        # Remove extra whitespace
        text = ' '.join(text.split())

        # Remove extremely long tokens
        words = text.split()
        words = [w for w in words if len(w) <= 50]

        return ' '.join(words)

    def combine_for_embeddings(self, row: Dict) -> str:
        parts = []

        if pd.notna(row.get('title')):
            parts.append(str(row['title']))

        if pd.notna(row.get('description')):
            parts.append(str(row['description']))

        if pd.notna(row.get('skills_desc')):
            parts.append(str(row['skills_desc']))

        if pd.notna(row.get('formatted_experience_level')):
            parts.append(str(row['formatted_experience_level']))

        if pd.notna(row.get('formatted_work_type')):
            parts.append(str(row['formatted_work_type']))

        combined = '. '.join(parts)

        return self.clean_text(combined)

Apply Preprocessing Methods

In [ ]:
jobs = pd.read_csv('../jobs_updated.csv')

In [ ]:
jobs_after = jobs.copy(deep=True)

In [ ]:
PREPROCESSOR = TextPreprocessor()

In [ ]:
jobs_after["combined_text"] = jobs_after.apply(
    lambda row: PREPROCESSOR.combine_for_embeddings(row),
    axis=1
)
jobs_after["skills_desc"] = jobs_after["skills_desc"].apply(
    lambda x: x[:1000] if isinstance(x, str) else x
)

In [ ]:
print(f"\nEmbedding text statistics:")
jobs_after['embedding_length'] = jobs_after['embedding_text'].str.len()
print(f"  Mean length: {jobs_after['embedding_length'].mean():.0f} characters")
print(f"  Median length: {jobs_after['embedding_length'].median():.0f} characters")
print(f"  Min length: {jobs_after['embedding_length'].min():.0f} characters")
print(f"  Max length: {jobs_after['embedding_length'].max():.0f} characters")